# Fine-tuning LoRA — agent de triaj ITSM

Rulează în **Google Colab, GPU T4 gratuit** (Runtime → Change runtime type → T4 GPU). Durează ~15–25 minute cu tot cu exportul.

1. Încarcă `train.jsonl` și `test.jsonl` din folderul `finetune/` al proiectului (iconița de folder din stânga → Upload).
2. Rulează celulele pe rând.
3. La final descarci `triaj-lora.Q4_K_M.gguf` și îl pui în `finetune/` pe laptop, apoi `ollama create triaj-lora -f finetune/Modelfile`.

Model de bază: **Llama 3.2 3B Instruct** — același pe care îl ai în Ollama ca `llama3.2`, ca comparația few-shot vs LoRA să fie pe același model.

In [ ]:
%%capture
!pip install unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_LEN = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=MAX_LEN,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")
train = load_dataset("json", data_files="train.jsonl", split="train")
train = train.map(lambda b: {"text": [tokenizer.apply_chat_template(m, tokenize=False) for m in b["messages"]]},
                  batched=True)
print(len(train), "exemple")
print(train[0]["text"][-400:])

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_LEN,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_steps=5,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)
# invata doar raspunsul (JSON-ul), nu si promptul
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)
stats = trainer.train()
print(stats)

## Verificare rapidă pe setul de test (în Colab)
Evaluarea oficială se face cu `scripts/evaluate.py` după import în Ollama; asta e doar un sanity check.

In [ ]:
import json
from sklearn.metrics import accuracy_score, f1_score

FastLanguageModel.for_inference(model)
test = [json.loads(l) for l in open("test.jsonl")]
y_true, y_pred, cat_true, cat_pred = [], [], [], []
for ex in test:
    msgs = ex["messages"][:2]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=60, do_sample=False)
    text = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    gold = json.loads(ex["messages"][2]["content"])
    try:
        pred = json.loads(text)
    except Exception:
        pred = {}
    y_true.append(gold["priority"]); y_pred.append(pred.get("priority", "INVALID"))
    cat_true.append(gold["category"]); cat_pred.append(pred.get("category", "INVALID"))

print("prioritate acc:", round(accuracy_score(y_true, y_pred), 3),
      "macro-F1:", round(f1_score(y_true, y_pred, labels=["P1","P2","P3","P4"], average="macro", zero_division=0), 3))
print("categorie acc:", round(accuracy_score(cat_true, cat_pred), 3))

## Export pentru Ollama (GGUF, cuantizat Q4_K_M)

In [ ]:
model.save_pretrained("triaj-lora-adapter")        # adaptorul LoRA (mic, ~50 MB)
tokenizer.save_pretrained("triaj-lora-adapter")
model.save_pretrained_gguf("triaj-lora", tokenizer, quantization_method="q4_k_m")
!ls -la triaj-lora*

In [ ]:
import glob, shutil
from google.colab import files
gguf = sorted(glob.glob("triaj-lora*/*.gguf") + glob.glob("*.gguf"), key=len)[-1]
shutil.copy(gguf, "triaj-lora.Q4_K_M.gguf")
files.download("triaj-lora.Q4_K_M.gguf")